# Data versioning & lineage — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def rate(x): return float(np.mean(np.asarray(x)))
def nearest_predict(train_x, train_y, test_x):
    train_x=np.asarray(train_x); train_y=np.asarray(train_y); preds=[]
    for t in np.asarray(test_x):
        preds.append(train_y[int(np.argmin(np.abs(train_x-t)))])
    return np.asarray(preds)
print('setup ok')

## Knowing exactly which data produced a model
Versioning and lineage make datasets reproducible artifacts. These examples use hashes, diffs, transform records, rollback, and impact tracing to show why data needs history like code.

In [ ]:
# 1) A content hash changes when one row changes
import hashlib
a=np.array([1,2,3]); b=np.array([1,2,4])
h=lambda x: hashlib.sha1(x.tobytes()).hexdigest()[:8]
plt.figure(figsize=(6,3)); plt.bar(['v1','v2'],[int(h(a),16)%1000,int(h(b),16)%1000]); plt.title(f'hashes {h(a)} vs {h(b)}')
assert h(a)!=h(b)

In [ ]:
# 2) Dataset diffs tell what changed, not only that it changed
old=np.array([1,2,3,4]); new=np.array([1,2,4,5]); added=set(new)-set(old); removed=set(old)-set(new)
plt.figure(figsize=(6,3)); plt.bar(['added','removed','kept'],[len(added),len(removed),len(set(old)&set(new))]); plt.title('row-level version diff')
assert added=={5} and removed=={3}

In [ ]:
# 3) Lineage records each transform in a reproducible chain
raw=np.array([1.,2.,3.]); cleaned=raw[raw>=2]; featured=cleaned**2
plt.figure(figsize=(6,3)); plt.plot([len(raw),len(cleaned),len(featured)],'-o'); plt.xticks([0,1,2],['raw','cleaned','featured']); plt.title('lineage stages and row counts')
assert np.array_equal(featured,np.array([4.,9.]))

In [ ]:
# 4) Rollback selects the last passing version
versions=np.array([1,2,3,4]); quality=np.array([.91,.93,.62,.94]); passing=versions[quality>=.9]
plt.figure(figsize=(6,3)); plt.plot(versions,quality,'-o'); plt.axhline(.9,c='r',ls='--'); plt.title('version 3 fails quality gate')
assert passing[-2]==2 and passing[-1]==4

In [ ]:
# 5) Impact tracing finds which model consumed a bad snapshot
uses={'model_A':'data_v1','model_B':'data_v2','model_C':'data_v2'}; impacted=[m for m,v in uses.items() if v=='data_v2']
plt.figure(figsize=(6,3)); plt.bar(list(uses.keys()),[v=='data_v2' for v in uses.values()]); plt.title('models depending on data_v2')
assert impacted==['model_B','model_C']